In [0]:
bronze_layer_path = "/Volumes/brazilian_e-commerce_olist_project/default/e-commerce_datasets/Bronze/"

bronze_customers_df = spark.read.format("delta").load(
    f"{bronze_layer_path}/customers"
)

bronze_orders_df = spark.read.format("delta").load(
    f"{bronze_layer_path}/orders"
)

bronze_order_items_df = spark.read.format("delta").load(
    f"{bronze_layer_path}/order_items"
)

bronze_products_df = spark.read.format("delta").load(
    f"{bronze_layer_path}/products"
)

verifying our data


In [0]:
bronze_customers_df.count()

99441

In [0]:
bronze_orders_df.count()

99441

In [0]:
bronze_order_items_df.count()

112650

In [0]:
bronze_products_df.count()

32951

# **acctual transfromation is starting from here**

### Cleaning Customers data

In [0]:
from pyspark.sql.functions import col, sum

bronze_customers_df.select(
    [sum(col(c).isNull().cast('int')).alias(c) for c in bronze_customers_df.columns]
).display()


customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,source_file,ingestion_timestamp
0,0,0,0,0,0,0


In [0]:
bronze_customers_df.distinct().count()

99441

In [0]:
bronze_customers_df.limit(10).display()

customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,source_file,ingestion_timestamp
06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP,dbfs:/Volumes/brazilian_e-commerce_olist_project/default/e-commerce_datasets/Raw/olist_customers_dataset.csv,2026-09-20T10:29:53.494Z
18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP,dbfs:/Volumes/brazilian_e-commerce_olist_project/default/e-commerce_datasets/Raw/olist_customers_dataset.csv,2026-09-20T10:29:53.494Z
4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP,dbfs:/Volumes/brazilian_e-commerce_olist_project/default/e-commerce_datasets/Raw/olist_customers_dataset.csv,2026-09-20T10:29:53.494Z
b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP,dbfs:/Volumes/brazilian_e-commerce_olist_project/default/e-commerce_datasets/Raw/olist_customers_dataset.csv,2026-09-20T10:29:53.494Z
4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP,dbfs:/Volumes/brazilian_e-commerce_olist_project/default/e-commerce_datasets/Raw/olist_customers_dataset.csv,2026-09-20T10:29:53.494Z
879864dab9bc3047522c92c82e1212b8,4c93744516667ad3b8f1fb645a3116a4,89254,jaragua do sul,SC,dbfs:/Volumes/brazilian_e-commerce_olist_project/default/e-commerce_datasets/Raw/olist_customers_dataset.csv,2026-09-20T10:29:53.494Z
fd826e7cf63160e536e0908c76c3f441,addec96d2e059c80c30fe6871d30d177,4534,sao paulo,SP,dbfs:/Volumes/brazilian_e-commerce_olist_project/default/e-commerce_datasets/Raw/olist_customers_dataset.csv,2026-09-20T10:29:53.494Z
5e274e7a0c3809e14aba7ad5aae0d407,57b2a98a409812fe9618067b6b8ebe4f,35182,timoteo,MG,dbfs:/Volumes/brazilian_e-commerce_olist_project/default/e-commerce_datasets/Raw/olist_customers_dataset.csv,2026-09-20T10:29:53.494Z
5adf08e34b2e993982a47070956c5c65,1175e95fb47ddff9de6b2b06188f7e0d,81560,curitiba,PR,dbfs:/Volumes/brazilian_e-commerce_olist_project/default/e-commerce_datasets/Raw/olist_customers_dataset.csv,2026-09-20T10:29:53.494Z
4b7139f34592b3a31687243a302fa75b,9afe194fb833f79e300e37e580171f22,30575,belo horizonte,MG,dbfs:/Volumes/brazilian_e-commerce_olist_project/default/e-commerce_datasets/Raw/olist_customers_dataset.csv,2026-09-20T10:29:53.494Z


#### First checking for our city and state name's case is only upper or lower or mixed

In [0]:
from pyspark.sql.functions import col, lower, upper, lit
[col(i) == "SP" for i in bronze_customers_df.columns if i=='customer_state']

#checking for upper case only
bronze_customers_df.filter(col('customer_state') == upper(col('customer_state'))).select("customer_state").count()
#checking for lower case only
bronze_customers_df.filter(col('customer_state') == lower(col('customer_state'))).select("customer_state").count()
#checking for mixed case only
bronze_customers_df.filter(
    ((col('customer_state') != upper(col('customer_state'))) & (col('customer_state') != lower(col('customer_state'))))

).select('customer_state').count()

#then if found cleaning there name in below cell

0

cleaning extra space from both and conveting them to a lower and upper case respectively

In [0]:
from pyspark.sql.functions import trim, lower, upper

silver_customers_df = (
    bronze_customers_df
    .withColumn("customer_city", lower(trim(col("customer_city"))))
    .withColumn("customer_state", upper(trim(col("customer_state"))))
)
silver_customers_df.select(
    "customer_id",
    "customer_city",
    "customer_state"
).limit(10).display()

customer_id,customer_city,customer_state
06b8999e2fba1a1fbc88172c00ba8bc7,franca,SP
18955e83d337fd6b2def6b18a428ac77,sao bernardo do campo,SP
4e7b3e00288586ebd08712fdd0374a03,sao paulo,SP
b2b6027bc5c5109e529d4dc6358b12c3,mogi das cruzes,SP
4f2d8ab171c80ec8364f7c12e35b23ad,campinas,SP
879864dab9bc3047522c92c82e1212b8,jaragua do sul,SC
fd826e7cf63160e536e0908c76c3f441,sao paulo,SP
5e274e7a0c3809e14aba7ad5aae0d407,timoteo,MG
5adf08e34b2e993982a47070956c5c65,curitiba,PR
4b7139f34592b3a31687243a302fa75b,belo horizonte,MG


After transforming our customer dataset, now storing(saving) this file in silver layer

In [0]:
#this line is just replacing text with bronze_layer_path with silver at the end, so that we can use the same path for silver layer
silver_layer_path=bronze_layer_path.replace("Bronze", "Silver")

(silver_customers_df.write
    .format("delta") 
    .mode("overwrite") 
    .save(f"{silver_layer_path}/customers")
)

Just verifying if customer data is sucesfully loaded into silver layer or not

In [0]:
silver_customers_df = spark.read.format('delta').load((f"{silver_layer_path}/customers"))
# silver_customers_df.display()

### Cleaning Orders data

In [0]:
# bronze_orders_df.display()

First we are checking null values in orders bronze dataframe

In [0]:
from pyspark.sql.functions import col, sum

bronze_orders_df.select(
    [sum(col(c).isNull().cast("int")).alias(c)
     for c in bronze_orders_df.columns]
).display()

order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,source_file,ingestion_timestamp
0,0,0,0,160,1783,2965,0,0,0


In [0]:
bronze_orders_df.select("order_id").distinct().count()

99441

In [0]:
from pyspark.sql.functions import to_timestamp

silver_orders_df = (
    bronze_orders_df
    .withColumn("order_purchase_timestamp",
                to_timestamp("order_purchase_timestamp"))
    .withColumn("order_approved_at",
                to_timestamp("order_approved_at"))
    .withColumn("order_delivered_carrier_date",
                to_timestamp("order_delivered_carrier_date"))
    .withColumn("order_delivered_customer_date",
                to_timestamp("order_delivered_customer_date"))
    .withColumn("order_estimated_delivery_date",
                to_timestamp("order_estimated_delivery_date"))
)
# silver_orders_df.display()

In [0]:
from pyspark.sql.functions import count
different_types_of_ordres=silver_orders_df.groupBy("order_status").agg(count("*").alias("total"))
different_types_of_ordres.display()

order_status,total
delivered,96478
invoiced,314
shipped,1107
processing,301
unavailable,609
canceled,625
created,5
approved,2


In [0]:
from pyspark.sql.functions import to_date

silver_orders_df = silver_orders_df.withColumn(
    "order_purchase_date",
    to_date("order_purchase_timestamp")
)

#verifying it
silver_orders_df.select(
    "order_id",
    "order_purchase_timestamp",
    "order_purchase_date"
).limit(10).display()

order_id,order_purchase_timestamp,order_purchase_date
e481f51cbdc54678b7cc49136f2d6af7,2017-10-02T10:56:33.000Z,2017-10-02
53cdb2fc8bc7dce0b6741e2150273451,2018-07-24T20:41:37.000Z,2018-07-24
47770eb9100c2d0c44946d9cf07ec65d,2018-08-08T08:38:49.000Z,2018-08-08
949d5b44dbf5de918fe9c16f97b45f8a,2017-11-18T19:28:06.000Z,2017-11-18
ad21c59c0840e6cb83a9ceb5573f8159,2018-02-13T21:18:39.000Z,2018-02-13
a4591c265e18cb1dcee52889e2d8acc3,2017-07-09T21:57:05.000Z,2017-07-09
136cce7faa42fdb2cefd53fdc79a6098,2017-04-11T12:22:08.000Z,2017-04-11
6514b8ad8028c9f2cc2374ded245783f,2017-05-16T13:10:30.000Z,2017-05-16
76c6e866289321a7c93b82b54852dc33,2017-01-23T18:29:09.000Z,2017-01-23
e69bfb5eb88e0ed6a785585b27e16dbf,2017-07-29T11:55:02.000Z,2017-07-29


Now saving ordes data into silver layer

In [0]:
silver_orders_df.write.format("delta").mode("overwrite").save(f"{silver_layer_path}/orders")

In [0]:
spark.read.format("delta").load(f"{silver_layer_path}/orders").count()

99441

### Cleaning Orders_item data

In [0]:
bronze_order_items_df.select(
    [sum(col(c).isNull().cast("int")).alias(c)
     for c in bronze_order_items_df.columns]
).display()

order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,source_file,ingestion_timestamp
0,0,0,0,0,0,0,0,0


checking duplicate records, for both primary key and foreign key ==> 
order_item_id and order_id

In [0]:
bronze_order_items_df.select(
    "order_id",
    "order_item_id"
).distinct().count()

112650

In [0]:
bronze_order_items_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- source_file: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)



In [0]:
bronze_orders_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- source_file: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)



everything of orders_item file is correct as of now so simply saving it in silver layer


In [0]:
silver_order_items_df = bronze_order_items_df
silver_order_items_df.write.format("delta").mode("overwrite").save(f"{silver_layer_path}/order_items")

In [0]:
silver_order_items_df=spark.read.format("delta").load(f"{silver_layer_path}/order_items").count()

### Cleaning Products data

In [0]:
bronze_products_df.limit(10).display()

product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,source_file,ingestion_timestamp
1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40,287,1,225,16,10,14,dbfs:/Volumes/brazilian_e-commerce_olist_project/default/e-commerce_datasets/Raw/olist_products_dataset.csv,2026-09-20T10:30:03.476Z
3aa071139cb16b67ca9e5dea641aaa2f,artes,44,276,1,1000,30,18,20,dbfs:/Volumes/brazilian_e-commerce_olist_project/default/e-commerce_datasets/Raw/olist_products_dataset.csv,2026-09-20T10:30:03.476Z
96bd76ec8810374ed1b65e291975717f,esporte_lazer,46,250,1,154,18,9,15,dbfs:/Volumes/brazilian_e-commerce_olist_project/default/e-commerce_datasets/Raw/olist_products_dataset.csv,2026-09-20T10:30:03.476Z
cef67bcfe19066a932b7673e239eb23d,bebes,27,261,1,371,26,4,26,dbfs:/Volumes/brazilian_e-commerce_olist_project/default/e-commerce_datasets/Raw/olist_products_dataset.csv,2026-09-20T10:30:03.476Z
9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37,402,4,625,20,17,13,dbfs:/Volumes/brazilian_e-commerce_olist_project/default/e-commerce_datasets/Raw/olist_products_dataset.csv,2026-09-20T10:30:03.476Z
41d3672d4792049fa1779bb35283ed13,instrumentos_musicais,60,745,1,200,38,5,11,dbfs:/Volumes/brazilian_e-commerce_olist_project/default/e-commerce_datasets/Raw/olist_products_dataset.csv,2026-09-20T10:30:03.476Z
732bd381ad09e530fe0a5f457d81becb,cool_stuff,56,1272,4,18350,70,24,44,dbfs:/Volumes/brazilian_e-commerce_olist_project/default/e-commerce_datasets/Raw/olist_products_dataset.csv,2026-09-20T10:30:03.476Z
2548af3e6e77a690cf3eb6368e9ab61e,moveis_decoracao,56,184,2,900,40,8,40,dbfs:/Volumes/brazilian_e-commerce_olist_project/default/e-commerce_datasets/Raw/olist_products_dataset.csv,2026-09-20T10:30:03.476Z
37cc742be07708b53a98702e77a21a02,eletrodomesticos,57,163,1,400,27,13,17,dbfs:/Volumes/brazilian_e-commerce_olist_project/default/e-commerce_datasets/Raw/olist_products_dataset.csv,2026-09-20T10:30:03.476Z
8c92109888e8cdf9d66dc7e463025574,brinquedos,36,1156,1,600,17,10,12,dbfs:/Volumes/brazilian_e-commerce_olist_project/default/e-commerce_datasets/Raw/olist_products_dataset.csv,2026-09-20T10:30:03.476Z


In [0]:
from pyspark.sql.functions import col, sum
bronze_products_df.select(
    [sum(col(c).isNull().cast("int")).alias(c) for c in bronze_products_df.columns]
).display()

product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,source_file,ingestion_timestamp
0,610,610,610,610,2,2,2,2,0,0


In [0]:
bronze_products_df.select("product_id").distinct().count()

32951

In [0]:
bronze_products_df.count()

32951

Handling null values 

In [0]:
from pyspark.sql.functions import nvl, lit
silver_product_df = bronze_products_df

silver_product_df=silver_product_df.fillna(
    {
        "product_category_name": "unknown"
    }
)
silver_product_df.limit(10).display()

product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,source_file,ingestion_timestamp
1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40,287,1,225,16,10,14,dbfs:/Volumes/brazilian_e-commerce_olist_project/default/e-commerce_datasets/Raw/olist_products_dataset.csv,2026-09-20T10:30:03.476Z
3aa071139cb16b67ca9e5dea641aaa2f,artes,44,276,1,1000,30,18,20,dbfs:/Volumes/brazilian_e-commerce_olist_project/default/e-commerce_datasets/Raw/olist_products_dataset.csv,2026-09-20T10:30:03.476Z
96bd76ec8810374ed1b65e291975717f,esporte_lazer,46,250,1,154,18,9,15,dbfs:/Volumes/brazilian_e-commerce_olist_project/default/e-commerce_datasets/Raw/olist_products_dataset.csv,2026-09-20T10:30:03.476Z
cef67bcfe19066a932b7673e239eb23d,bebes,27,261,1,371,26,4,26,dbfs:/Volumes/brazilian_e-commerce_olist_project/default/e-commerce_datasets/Raw/olist_products_dataset.csv,2026-09-20T10:30:03.476Z
9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37,402,4,625,20,17,13,dbfs:/Volumes/brazilian_e-commerce_olist_project/default/e-commerce_datasets/Raw/olist_products_dataset.csv,2026-09-20T10:30:03.476Z
41d3672d4792049fa1779bb35283ed13,instrumentos_musicais,60,745,1,200,38,5,11,dbfs:/Volumes/brazilian_e-commerce_olist_project/default/e-commerce_datasets/Raw/olist_products_dataset.csv,2026-09-20T10:30:03.476Z
732bd381ad09e530fe0a5f457d81becb,cool_stuff,56,1272,4,18350,70,24,44,dbfs:/Volumes/brazilian_e-commerce_olist_project/default/e-commerce_datasets/Raw/olist_products_dataset.csv,2026-09-20T10:30:03.476Z
2548af3e6e77a690cf3eb6368e9ab61e,moveis_decoracao,56,184,2,900,40,8,40,dbfs:/Volumes/brazilian_e-commerce_olist_project/default/e-commerce_datasets/Raw/olist_products_dataset.csv,2026-09-20T10:30:03.476Z
37cc742be07708b53a98702e77a21a02,eletrodomesticos,57,163,1,400,27,13,17,dbfs:/Volumes/brazilian_e-commerce_olist_project/default/e-commerce_datasets/Raw/olist_products_dataset.csv,2026-09-20T10:30:03.476Z
8c92109888e8cdf9d66dc7e463025574,brinquedos,36,1156,1,600,17,10,12,dbfs:/Volumes/brazilian_e-commerce_olist_project/default/e-commerce_datasets/Raw/olist_products_dataset.csv,2026-09-20T10:30:03.476Z


In [0]:
from pyspark.sql.functions import col, sum
x=silver_product_df.select(
    [sum(col(c).isNull().cast('int')).alias(c) for c in silver_product_df.columns]
)
x.display()


product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,source_file,ingestion_timestamp
0,0,610,610,610,2,2,2,2,0,0


now saving products data frame to silver layer 

In [0]:
silver_product_df.write.format("delta").mode("overwrite").save(f"{silver_layer_path}/products")

In [0]:
spark.read.format("delta").load(f"{silver_layer_path}/products").count()

32951